In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg

# Lecture 7 - Convolutional Neural Networks

Convolutional layers as filter banks, what the feature maps of a trained network look like at different depths, pooling, and how large the receptive field of a unit really is.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## A convolutional layer is a bank of filters

A convolutional layer with $C_{out}$ filters of size $k \times k \times C_{in}$ is exactly the filtering from Lecture 2, done $C_{out}$ times. Here we build one by hand with classic filters as its weights (`F.conv2d` computes a cross-correlation, i.e. it does not flip the kernel).

In [ ]:
image = cv2.cvtColor(cv2.imread('image.jpg'), cv2.COLOR_BGR2RGB)
gray = torch.from_numpy(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)).float()[None, None] / 255   # (batch, channels, H, W)

gauss = torch.from_numpy(np.outer(*[cv2.getGaussianKernel(5, 1.0)] * 2)).float()
filters = {
    'Gaussian blur': gauss,
    'Sobel x (vertical edges)': torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float(),
    'Sobel y (horizontal edges)': torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float(),
    'Laplacian': torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]]).float(),
}
# pad all kernels to 5x5 and stack them into the weight tensor of one layer: (C_out, C_in, k, k)
weight = torch.stack([F.pad(k, [(5 - k.shape[1]) // 2] * 2 + [(5 - k.shape[0]) // 2] * 2) for k in filters.values()])[:, None]
layer = torch.nn.Conv2d(1, len(filters), kernel_size=5, padding=2, bias=False)
layer.weight.data = weight
with torch.no_grad():
    out = layer(gray)
print('input', tuple(gray.shape), '-> output', tuple(out.shape), '| weights', tuple(layer.weight.shape))

fig, axs = plt.subplots(2, len(filters) + 1, figsize=(25, 7), gridspec_kw={'height_ratios': [1, 3]})
axs[0, 0].axis('off')
axs[1, 0].imshow(gray[0, 0], cmap='gray')
axs[1, 0].set_title('input')
for i, name in enumerate(filters):
    axs[0, i + 1].imshow(weight[i, 0], cmap='RdBu_r', vmin=-weight[i].abs().max(), vmax=weight[i].abs().max())
    axs[0, i + 1].set_title(name)
    v = np.percentile(out[0, i].abs().numpy(), 99) if i else 1   # robust colour scale
    axs[1, i + 1].imshow(out[0, i], cmap='gray' if i == 0 else 'RdBu_r', vmin=0 if i == 0 else -v, vmax=v)
for ax in axs.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
savefig('cnn_filter_bank')
plt.show()

In [ ]:
# output size = floor((input + 2 * padding - kernel) / stride) + 1
x = torch.zeros(1, 3, 224, 224)
for k, s, p in [(3, 1, 0), (3, 1, 1), (5, 1, 2), (3, 2, 1), (7, 2, 3), (2, 2, 0)]:
    y = torch.nn.Conv2d(3, 64, kernel_size=k, stride=s, padding=p)(x)
    print(f'kernel {k}, stride {s}, padding {p}: 224x224 -> {y.shape[2]}x{y.shape[3]}   ({3 * 64 * k * k + 64} parameters)')

# max pooling keeps the largest value in each window
x = torch.tensor([[1, 1, 2, 4], [5, 6, 7, 8], [3, 2, 1, 0], [1, 2, 3, 4]]).float()
print('\ninput:\n', x.int().numpy())
print('2x2 max pooling, stride 2:\n', F.max_pool2d(x[None, None], 2)[0, 0].int().numpy())
print('2x2 average pooling, stride 2:\n', F.avg_pool2d(x[None, None], 2)[0, 0].numpy())

## Feature maps of a trained network

A ResNet-18 trained on ImageNet: the spatial resolution halves at every stage while the number of channels doubles. Early channels respond to edges and colours, later ones to larger, more abstract patterns (and their maps become too coarse to see much).

In [ ]:
weights = torchvision.models.ResNet18_Weights.DEFAULT
resnet = torchvision.models.resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()

dog = cv2.cvtColor(cv2.imread('image_dog.jpg'), cv2.COLOR_BGR2RGB)
dog = dog[:, :dog.shape[0]]                                   # square crop on the left
x = preprocess(torch.from_numpy(dog).permute(2, 0, 1))[None].to(device)

stages = {}
with torch.no_grad():
    h = resnet.relu(resnet.bn1(resnet.conv1(x)))
    stages['conv1'] = h
    h = resnet.maxpool(h)
    for name in ['layer1', 'layer2', 'layer3', 'layer4']:
        h = getattr(resnet, name)(h)
        stages[name] = h
    print('prediction:', weights.meta['categories'][resnet.fc(torch.flatten(resnet.avgpool(h), 1)).argmax()])

n = 6
fig, axs = plt.subplots(len(stages), n + 1, figsize=(2.6 * (n + 1), 2.6 * len(stages)))
for row, (name, fmap) in zip(axs, stages.items()):
    fmap = fmap[0].cpu()
    strongest = fmap.mean(dim=(1, 2)).argsort(descending=True)[:n]   # the most active channels
    row[0].imshow(cv2.resize(dog, (224, 224)))
    row[0].set_title(f'{name}: {fmap.shape[0]} x {fmap.shape[1]} x {fmap.shape[2]}', fontsize=10, loc='left')
    for ax, c in zip(row[1:], strongest):
        ax.imshow(fmap[c], cmap='viridis')
    for ax in row:
        ax.set_xticks([])
        ax.set_yticks([])
savefig('cnn_feature_maps')
plt.show()

## Effective receptive field

The *theoretical* receptive field of a unit (the input region that can influence it) grows with every layer (Sheet 3). How much does each input pixel actually influence it? We compute the gradient of one unit in the centre of a feature map with respect to the input image. The influence is concentrated in the middle and falls off smoothly: the *effective* receptive field is much smaller than the theoretical one (Luo et al., 2016), and training makes it larger.

In [ ]:
def effective_receptive_field(model, layer_name, n_images=16, size=320):
    model.eval()
    grads = torch.zeros(size, size)
    rng = np.random.default_rng(0)
    for _ in range(n_images):
        # random crops of the two photos as inputs
        src = dog if rng.random() < 0.5 else image
        y0, x0 = rng.integers(0, src.shape[0] - size), rng.integers(0, src.shape[1] - size)
        crop = torch.from_numpy(np.ascontiguousarray(src[y0:y0+size, x0:x0+size])).permute(2, 0, 1).float() / 255
        x = ((crop - torch.tensor([0.485, 0.456, 0.406])[:, None, None]) / torch.tensor([0.229, 0.224, 0.225])[:, None, None])[None].to(device)
        x.requires_grad_(True)
        h = model.maxpool(model.relu(model.bn1(model.conv1(x))))
        for name in ['layer1', 'layer2', 'layer3', 'layer4']:
            h = getattr(model, name)(h)
            if name == layer_name:
                break
        cy, cx = h.shape[2] // 2, h.shape[3] // 2
        h[0, :, cy, cx].sum().backward()
        grads += x.grad[0].abs().sum(0).cpu()
    return grads / grads.max()

random_resnet = torchvision.models.resnet18(weights=None).to(device)
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for row, (model, label) in zip(axs, [(random_resnet, 'random weights'), (resnet, 'trained')]):
    for ax, layer_name in zip(row, ['layer1', 'layer2', 'layer3', 'layer4']):
        erf = effective_receptive_field(model, layer_name)
        ax.imshow(erf ** 0.5, cmap='magma')
        ax.set_title(f'{layer_name}, {label}')
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle('Effective receptive field: |gradient| of the central unit w.r.t. the 320x320 input (square root for visibility)')
savefig('cnn_effective_receptive_field')
plt.show()